In [2]:
import geopandas as gpd
from shapely.ops import unary_union
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path
import matplotlib.patches as mpatches

# === Input Paths ===
obs_path = "/Volumes/External/TJ_SAR/01_data/shapefiles/plumeExtentv4.shp"
pred_base_path = "/Volumes/External/TJ_SAR/06_shapefiles/algorithmOutputperStudyArea"

# === Output Directory ===
output_dir = "/Volumes/External/TJ_SAR/05_viz/IoUplots"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# === Load Observed Shapefile and Reproject ===
obs_gdf = gpd.read_file(obs_path)
obs_gdf = obs_gdf.to_crs(epsg=32611)  # UTM Zone 11N

# === Store Results ===
results = []

# === Loop Through Observed Features ===
for idx, row in obs_gdf.iterrows():
    name = f"{row['sourceName']}_pre_mask_algo_{row['outflow']}"
    pred_path = os.path.join(pred_base_path, f"{name}.shp")

    if os.path.exists(pred_path):
        try:
            # Load and reproject predicted shapefile
            pred_gdf = gpd.read_file(pred_path)
            pred_gdf = pred_gdf.to_crs(epsg=32611)

            # Prepare geometries
            obs_geom = unary_union([row.geometry])
            pred_geom = unary_union([geom for geom in pred_gdf.geometry if geom.is_valid])

            # IoU calculation
            intersection = obs_geom.intersection(pred_geom).area
            union = obs_geom.union(pred_geom).area
            iou = intersection / union if union > 0 else 0

            # Area in square kilometers
            obs_area_km2 = obs_geom.area / 1e6
            pred_area_km2 = pred_geom.area / 1e6

            # Cleaned name
            clean_name = name.replace("_pre_mask_algo_", "_")

            # Append result
            results.append({
                "index": idx + 1,
                "name": name,
                "clean_name": clean_name,
                "obs_geom": obs_geom,
                "pred_geom": pred_geom,
                "iou": iou,
                "obs_area_km2": obs_area_km2,
                "pred_area_km2": pred_area_km2
            })

        except Exception as e:
            print(f"Error processing {name}: {e}")
            continue

# === Generate Overlay Plots ===
for result in results:
    fig, ax = plt.subplots()
    gpd.GeoSeries(result["obs_geom"]).plot(ax=ax, color="blue", alpha=0.5)
    gpd.GeoSeries(result["pred_geom"]).plot(ax=ax, color="red", alpha=0.5)

    # Legend
    obs_patch = mpatches.Patch(color='blue', alpha=0.5, label='Observed')
    pred_patch = mpatches.Patch(color='red', alpha=0.5, label='Predicted')
    plt.legend(handles=[obs_patch, pred_patch])

    # Title and save
    plt.title(f"{result['name']}\nIoU: {result['iou']:.4f}")
    plot_path = os.path.join(output_dir, f"{result['index']:02d}_{result['name']}_iou.png")
    plt.savefig(plot_path)
    plt.close()

# === Create DataFrame and Export CSV ===
iou_df = pd.DataFrame(results)[[
    "index", "name", "clean_name", "iou", "obs_area_km2", "pred_area_km2"
]]
iou_df.sort_values(by="index", inplace=True)

# Stats
valid_ious = iou_df["iou"].dropna()
mean_iou = valid_ious.mean()
median_iou = valid_ious.median()

print(f"\nMean IoU: {mean_iou:.4f}")
print(f"Median IoU: {median_iou:.4f}")

# Export CSV
output_csv_path = os.path.join(output_dir, "iou_summary.csv")
iou_df.to_csv(output_csv_path, index=False)
print(f"IoU results written to: {output_csv_path}")




Mean IoU: 0.3630
Median IoU: 0.3401
IoU results written to: /Volumes/External/TJ_SAR/05_viz/IoUplots/iou_summary.csv
